# Step 1: 수동 메모리 주입 (Middleware 방식)

이 예제에서는 에이전트의 단기 메모리 한계를 극복하기 위해 가장 기초적인 방법인 **미들웨어(Middleware)를 통한 시스템 프롬프트 주입 방식**을 배웁니다.

* **특징**: LLM이 직접 메모리를 다루지 않고, 백그라운드에서 개발자가 강제로 정보를 넣어줍니다.
* **한계**: 대화 중 새로운 사실을 스스로 기억(Write)할 수 없습니다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### 1. 장기 메모리 저장소(Store) 준비

In [ ]:
from langgraph.store.memory import InMemoryStore

# 임베딩(검색 기능) 없이 순수한 키-밸류 저장소로 초기화합니다.
store = InMemoryStore()

In [ ]:
# 사용자 및 앱 컨텍스트(네임스페이스) 정의
user_id = "user_001"
application_context = "personal_assistant"
namespace = (user_id, application_context)

### 2. 수동으로 기억(Memory) 저장하기
LLM이 스스로 저장하는 도구가 없으므로, 개발자가 코드 레벨에서 직접 데이터를 넣어줍니다.

In [ ]:
store.put(
    namespace,
    "memory_001",
    {
        "facts": [
            "사용자는 아아를 선호함",
            "사용자는 매일 아침 7시에 일어남",
        ],
        "language": "Korean",
    },
)

In [ ]:
store.put(
    namespace,
    "memory_002",
    {
        "facts": [
            "사용자는 랭체인 공부를 좋아함",
            "학습용 챗봇 프로젝트를 진행 중",
        ]
    },
)

### 3. 미들웨어(Middleware)로 메모리 주입하기
LLM이 응답을 생성하기 직전에 가로채서(intercept), Store에 있는 기억을 꺼내어 시스템 프롬프트에 텍스트로 강제 주입하는 역할을 합니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str
    app_name: str

In [ ]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_memory(request, handler):
    # 1. 현재 사용자의 네임스페이스에 해당하는 메모리를 모두 가져옵니다.
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name
    memories = request.runtime.store.search((current_user, current_app))

    memory_content = "기록된 정보 없음"
    if memories:
        extracted_facts = []
        for item in memories:
            if "facts" in item.value:
                extracted_facts.extend(item.value["facts"])
        memory_content = "\n- ".join(extracted_facts)

    # 2. 시스템 프롬프트(System Prompt)에 메모리를 텍스트로 강제 주입합니다.
    system_message=f"사용자 관련 장기 메모리 : {memory_content}"
    request = request.override(system_prompt=system_message)
    
    # 3. 주입이 끝난 후 LLM 호출을 계속 진행합니다.
    return handler(request)

### 4. 에이전트 생성 및 실행

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    store=store, # 저장소 연결
    context_schema=Context,
    middleware=[inject_memory] # 우리가 만든 강제 주입 미들웨어 장착
)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response